In [17]:
import pandas as pd
import glob

# ============================
# 1. Load Market Daily Prices
# ============================

# Keep 2014–2024 only
market = pd.read_csv("../data/market/all_prices_2014_2023.csv") 
# ============================
# 2. Load Macro (daily, ffilled)
# ============================

macro = pd.read_parquet("../data/model/macro_features.parquet")
macro.index = pd.to_datetime(macro.index)
macro = macro.sort_index()

# ============================
# 3. Load Fundamentals (quarterly)
# ============================

fund = pd.read_csv("../data/model/fundamental_features.csv")  
fund["datadate"] = pd.to_datetime(fund["datadate"])
fund = fund.sort_values(["tic", "datadate"])

# --- Convert quarterly Compustat to daily using forward-fill ---

fund = fund.copy()
fund["datadate"] = pd.to_datetime(fund["datadate"])

# 1. Sort to ensure correct forward-fill
fund = fund.sort_values(["tic", "datadate"])

# 2. Convert quarterly to daily
fund_daily = (
    fund
        .set_index("datadate")
        .groupby("tic", group_keys=False)
        .resample("D")          # daily frequency
        .ffill()                # forward-fill values until next 10-Q
        .reset_index()
)

# 3. Rename for merge compatibility
fund_daily = fund_daily.rename(columns={"datadate": "Date"})

# 4. Restrict to your modeling window (safe window for fundamentals)
fund_daily = fund_daily[
    (fund_daily["Date"] >= "2013-12-01") &
    (fund_daily["Date"] <= "2024-12-31")
]

# rename market 'Ticker' to 'tic' for merge
market = market.rename(columns={"Ticker": "tic"})
market["Date"] = pd.to_datetime(market["Date"], utc=True, errors="coerce")
market["Date"] = market["Date"].dt.tz_convert(None)
# keep only date
market["Date"] = market["Date"].dt.date
market["Date"] = pd.to_datetime(market["Date"])

# ============================
# 4. Merge: Market + Fundamentals
# ============================

df = market.merge(
    fund_daily,
    on=["tic", "Date"],
    how="left"
)



# ============================
# 5. Merge: Add Macro (daily)
# ============================

df = df.merge(
    macro,
    left_on="Date",
    right_index=True,
    how="left"
)


# ============================
# 6. Create Labels (Next-Day Return)
# ============================

df = df.sort_values(["tic", "Date"])
df["next_close"] = df.groupby("tic")["Close"].shift(-1)
df["return_next_day"] = (df["next_close"] - df["Close"]) / df["Close"]

# Clean final
df = df.drop(columns=["next_close"])


# ============================
# 7. Train / Validation / Test Split
# ============================

train = df[(df["Date"] >= "2016-01-01") & (df["Date"] < "2022-01-01")]
val   = df[(df["Date"] >= "2022-01-01") & (df["Date"] < "2023-01-01")]
test  = df[(df["Date"] >= "2023-01-01")]

# ============================
# 8. Save
# ============================

train.to_parquet("../data/model/combine_train.parquet")
val.to_parquet("../data/model/combine_val.parquet")
test.to_parquet("../data/model/combine_test.parquet")

print(train.shape, val.shape, test.shape)


/var/folders/c2/pprfn7j56vs360hbrz08yqkh0000gn/T/ipykernel_99108/2102988599.py:40: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .ffill()                # forward-fill values until next 10-Q


(729659, 40) (124747, 40) (124727, 40)


In [18]:
# load news

def  load_news_data(path):
    news = pd.read_parquet(path)
    news = news.rename(columns={"Stock_symbol": "tic"})

    news["Date"] = pd.to_datetime(news["Date"]).dt.tz_localize(None)

    news = (
        news.sort_values(["tic", "Date"])
            .ffill()
    )
    return news

train_news_path = "../data/model/train_news_daily_pca.parquet"
val_news_path = "../data/model/val_news_daily_pca.parquet"
test_news_path = "../data/model/test_news_daily_pca.parquet"
train_news = load_news_data(train_news_path)
val_news = load_news_data(val_news_path)
test_news = load_news_data(test_news_path)
print(train_news.shape, val_news.shape, test_news.shape)

(398780, 71) (46756, 71) (55339, 71)


In [19]:
train = train.merge(train_news, on=["Date", "tic"], how="left")
val   = val.merge(val_news, on=["Date", "tic"], how="left")
test  = test.merge(test_news, on=["Date", "tic"], how="left")


In [20]:
def fill_missing_values(df):

    # fill missing values

    pca_cols = [c for c in df.columns if c.startswith("pca_emb_")]

    # Force all PCA columns to numeric (NaN becomes NaN)
    df[pca_cols] = df[pca_cols].apply(pd.to_numeric, errors="coerce")


    news_cols = (
        ["mean_sentiment", "max_sentiment", "min_sentiment",
        "sum_sentiment", "news_count"] + pca_cols
    )

    df[news_cols] = df[news_cols].fillna(0)



    # -------- Growth (0)
    growth_cols = [
        "sales_growth_qoq","sales_growth_ttm",
        "asset_growth","equity_growth"
    ]
    df[growth_cols] = df[growth_cols].fillna(0)

    # -------- Profitability (0)
    profit_cols = [
        "roa_ttm","roe_ttm","gross_margin_ttm",
        "oper_margin_ttm","net_margin_ttm"
    ]
    df[profit_cols] = df[profit_cols].fillna(0)

    # -------- Valuation (0)
    val_cols = [
        "log_mktcap","bm","earnings_yield",
        "cf_yield","sales_yield","div_yield"
    ]
    df[val_cols] = df[val_cols].fillna(0)

    # -------- Risk & Liquidity
    df["leverage"] = df["leverage"].fillna(1.0)
    df["current_ratio"] = df["current_ratio"].fillna(1.0)
    df["cash_assets"] = df["cash_assets"].fillna(0.0)
    df["accruals_ta"] = df["accruals_ta"].fillna(0.0)

    # -------- News: already 0, but enforce safe fill
    news_cols = ["mean_sentiment","max_sentiment","min_sentiment",
                "sum_sentiment","news_count"] + \
                [c for c in df.columns if c.startswith("pca_emb_")]

    df[news_cols] = df[news_cols].fillna(0)

    return df


In [21]:

final_train = fill_missing_values(train)
final_val = fill_missing_values(val)
final_test = fill_missing_values(test)

# save
final_train.to_parquet("../data/model/final_train.parquet")
final_val.to_parquet("../data/model/final_val.parquet")
final_test.to_parquet("../data/model/final_test.parquet")



In [22]:
# y is return_next_day